In [16]:
# =====================================================================
# IMPORTS AND SETUP
# =====================================================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML

# Set Mapbox access token for satellite map (replace with your own token if needed)
px.set_mapbox_access_token('pk.eyJ1IjoiY2hyaWRkeXAiLCJhIjoiY2lxMnlsaGFiMDA4dXRubmtiZGthNzh1YiJ9.b9YxH3oO4km0fLH1zfLfg')


In [25]:
df.to_csv('Electric_Vehicle_Population_Data.csv', index=False)
df = pd.read_csv('Electric_Vehicle_Population_Data.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 278984 entries, 0 to 278983
Data columns (total 16 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   VIN (1-10)                                         278984 non-null  str    
 1   County                                             278984 non-null  str    
 2   City                                               278984 non-null  str    
 3   State                                              278984 non-null  str    
 4   Postal Code                                        278984 non-null  float64
 5   Model Year                                         278984 non-null  int64  
 6   Make                                               278984 non-null  str    
 7   Model                                              278984 non-null  str    
 8   Electric Vehicle Type                              278984 non-null  str    
 9   Clea

In [26]:
df.head()

,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 Census Tract
0,JN1AZ0CP5C,Stevens,Colville,WA,99114.0,2012,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,73.0,7.0,153331706,POINT (-117.90454 48.54657),AVISTA CORP,5.306595e+10
1,JTMABABA7P,Yakima,Yakima,WA,98903.0,2023,SUBARU,SOLTERRA,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,15.0,253586308,POINT (-120.71847 46.55029),PACIFICORP,5.307700e+10
2,1N4AZ1CP1J,King,Seattle,WA,98122.0,2018,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,151.0,37.0,333135022,POINT (-122.31009 47.60803),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303301e+10
3,5UX43EU09S,Kitsap,Poulsbo,WA,98370.0,2025,BMW,X5,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,40.0,23.0,267525737,POINT (-122.64681 47.73689),PUGET SOUND ENERGY INC,5.303594e+10
4,3C3CFFGE5F,Thurston,Yelm,WA,98597.0,2015,FIAT,500,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,87.0,2.0,474468501,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC,5.306701e+10


In [30]:
df.isnull().sum()

VIN (1-10)                                           0
County                                               0
City                                                 0
State                                                0
Postal Code                                          0
Model Year                                           0
Make                                                 0
Model                                                0
Electric Vehicle Type                                0
Clean Alternative Fuel Vehicle (CAFV) Eligibility    0
Electric Range                                       0
Legislative District                                 0
DOL Vehicle ID                                       0
Vehicle Location                                     0
Electric Utility                                     0
2020 Census Tract                                    0
dtype: int64

In [31]:
# ---------------------------------------------------------
# 0. LOAD DATA
# ---------------------------------------------------------
try:
    print("Loading Electric Vehicle Population Data...")
    df = pd.read_csv('Electric_Vehicle_Population_Data.csv')
    df.rename(columns={'Electric Vehicle Type': 'EV Type'}, inplace=True)
    print("Data loaded successfully!")
except Exception as e:
    print("Could not load data from 'data/Electric_Vehicle_Population_Data.csv'. Please check the path.", e)
    # Generate backup dummy data if needed

# ---------------------------------------------------------
# 1. THEME & GLOBAL CSS (ENHANCED AESTHETIC)
# ---------------------------------------------------------
BG_COLOR = "#0D1117"
CARD_COLOR = "#161B22"
ACCENT_COLOR = "#00D4FF"
SECONDARY_COLOR = "#FF6B6B"
TERTIARY_COLOR = "#4ECDC4"
TEXT_COLOR = "#E6EDF3"
HIGHLIGHT_COLOR = "#FFD93D"
BG = "#0f172a"          # dark background
CYAN = "#06b6d4"        # accent color
TEXT_PRI = "#f8fafc"    # primary text
TEXT_SEC = "#94a3b8"    # secondary text
BG_COLOR = "#020617"    # dashboard background

display(HTML(f"""
<style>
    .kpi-card {{
        background: linear-gradient(135deg, {CARD_COLOR} 0%, #1F2937 100%);
        border-radius: 15px;
        padding: 15px;
        margin: 12px 0;
        box-shadow: 0 8px 20px rgba(0,0,0,0.6);
        font-family: 'Segoe UI', Tahoma, sans-serif;
        border: 1px solid #30363D;
        transition: transform 0.2s;
    }}
    .kpi-card:hover {{ transform: translateY(-2px); }}
    .kpi-title {{ color: #8B949E; font-size: 12px; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 10px; font-weight: bold; }}
    .kpi-value {{ color: {ACCENT_COLOR}; font-size: 32px; font-weight: 900; text-shadow: 0 0 10px {ACCENT_COLOR}; }}
    .dash-title {{ color: {TEXT_COLOR}; font-size: 28px; font-weight: bold; padding-bottom: 15px; border-bottom: 3px solid {ACCENT_COLOR}; margin-bottom:20px; font-family: 'Segoe UI', Tahoma, sans-serif; }}
    .section-title {{ color: {CYAN}; font-size: 18px; font-weight: 600; margin: 30px 0 15px 0; font-family: 'Segoe UI', Tahoma, sans-serif; }}
    .desc-text {{ color: #8B949E; font-size: 15px; line-height: 1.7; font-family: 'Segoe UI', Tahoma, sans-serif; }}
    .bullet-list {{ color: #C9D1D9; font-size: 15px; line-height: 1.9; margin-left: -20px; font-family: 'Segoe UI', Tahoma, sans-serif; }}
    .insight-box {{ background: linear-gradient(135deg, #2D3748 0%, #4A5568 100%); border-radius: 12px; padding: 20px; margin: 15px 0; border-left: 5px solid {HIGHLIGHT_COLOR}; }}
    .insight-title {{ color: {HIGHLIGHT_COLOR}; font-weight: bold; margin-bottom: 10px; }}
    .insight-text {{ color: #E2E8F0; }}
    .tab-button {{ background: {CARD_COLOR}; color: {TEXT_COLOR}; border: 1px solid #30363D; border-radius: 8px; padding: 10px 20px; margin: 5px; cursor: pointer; transition: all 0.3s; }}
    .tab-button:hover {{ background: {ACCENT_COLOR}; color: {BG_COLOR}; }}
    .tab-button.active {{ background: {ACCENT_COLOR}; color: {BG_COLOR}; box-shadow: 0 0 15px {ACCENT_COLOR}; }}
</style>
"""))

# ---------------------------------------------------------
# 2. WIDGET DEFINITIONS (LEFT PANEL SLICERS)
# ---------------------------------------------------------
style = {'description_width': 'initial'}
layout_w = widgets.Layout(width='100%', margin='10px 0')

# Slicers
df = df.dropna(subset=['Make', 'Model Year', 'Electric Range']) # Clean up a bit for the widgets
all_makes = ['All'] + sorted(df['Make'].unique().tolist())
min_yr, max_yr = int(df['Model Year'].min()), int(df['Model Year'].max())

w_make = widgets.Dropdown(options=all_makes, value='All', description='🏭 Manufacturer:', style=style, layout=layout_w)
w_top_n = widgets.IntSlider(min=3, max=15, value=10,color=CYAN, description='🔟 Top N:', style=style, layout=layout_w)
w_year = widgets.IntRangeSlider(min=min_yr, max=max_yr,color=CYAN, value=[min_yr, max_yr], description='📅 Year Range:', style=style, layout=layout_w)

display(HTML("""
<style>
.widget-label {
    color: white !important;
    font-weight: 600;
}
</style>
"""))

kpi_html = widgets.HTML()

def update_kpis(dff):
    total = len(dff)
    top_make = dff['Make'].value_counts().index[0] if total > 0 else "N/A"
    avg_range = dff['Electric Range'].mean() if total > 0 else 0
    growth_rate = ((dff['Model Year'].max() - dff['Model Year'].min()) / max(dff['Model Year'].min(), 1)) * 100 if total > 0 else 0
    
    kpi_html.value = f"""
    <div class='kpi-card'>
        <div class='kpi-title'>🚗 Total Vehicles</div>
        <div class='kpi-value'>{total:,}</div>
    </div>
    <div class='kpi-card'>
        <div class='kpi-title'>🏆 Top Manufacturer</div>
        <div class='kpi-value'>{top_make}</div>
    </div>
    <div class='kpi-card'>
        <div class='kpi-title'>⚡ Avg. Range (Miles)</div>
        <div class='kpi-value'>{avg_range:.1f}</div>
    </div>
    <div class='kpi-card'>
        <div class='kpi-title'>📈 Growth Rate (%)</div>
        <div class='kpi-value'>{growth_rate:.1f}</div>
    </div>
    """

# ---------------------------------------------------------
# 3. CHART GENERATION ENGINE (12 VISUALS + MAP)
# ---------------------------------------------------------
chart_outputs = [widgets.Output() for _ in range(12)]
map_output = widgets.Output()
insights_output = widgets.Output()

pl_layout = dict(
    template='plotly_dark',
    paper_bgcolor=CARD_COLOR,
    plot_bgcolor=CARD_COLOR,
    margin=dict(l=20, r=20, t=60, b=20),
    font=dict(color=TEXT_COLOR, size=12),
    hoverlabel=dict(bgcolor=BG_COLOR, font_size=14, font_family='Segoe UI')
)

def build_charts(dff, top_n):
    figs = []
    
    if len(dff) == 0:
        empty = go.Figure().add_annotation(text="No data available for these filters.", showarrow=False, font=dict(size=18, color=TEXT_COLOR))
        empty.update_layout(**pl_layout, xaxis=dict(visible=False), yaxis=dict(visible=False))
        return [empty] * 12

    # PRE-AGGREGATION FOR PERFORMANCE
    make_counts = dff['Make'].value_counts().nlargest(top_n).reset_index()
    make_counts.columns = ['Make', 'Count']
    year_counts = dff.groupby('Model Year').size().reset_index(name='Count')
    dff_top = dff[dff['Make'].isin(make_counts['Make'].tolist())]

    # [1] Bar Chart: Top Manufacturers
    f1 = px.bar(make_counts, x='Make', y='Count', title="1. Top Manufacturers", color='Count', color_continuous_scale='Blues')
    f1.update_layout(**pl_layout, coloraxis_showscale=False)
    figs.append(f1)
    
    # [2] Pie Chart: EV Type Distribution
    if 'EV Type' in dff.columns:
        t_c = dff['EV Type'].value_counts().reset_index()
        t_c.columns = ['EV Type', 'Count']
        f2 = px.pie(t_c, names='EV Type', values='Count', title="2. Vehicle Type Distribution", hole=0.5, color_discrete_sequence=[ACCENT_COLOR, SECONDARY_COLOR])
        f2.update_traces(textposition='inside', textinfo='percent+label', marker=dict(line=dict(color=CARD_COLOR, width=2)))
        f2.update_layout(**pl_layout, showlegend=False)
        figs.append(f2)
    else: figs.append(go.Figure().update_layout(title="2. Vehicle Type Dist (N/A)", **pl_layout))

    # [3] Treemap: Market Share
    f3 = px.treemap(make_counts, path=[px.Constant("Market"), 'Make'], values='Count', title="3. Market Share Breakdown", color='Count', color_continuous_scale='Teal')
    f3.update_layout(**pl_layout, coloraxis_showscale=False)
    figs.append(f3)

    # [4] Line Chart: Registration Growth
    f4 = px.line(year_counts, x='Model Year', y='Count', title="4. Registration Growth YoY", markers=True)
    f4.update_traces(line_color=ACCENT_COLOR, line_width=4, marker=dict(size=10, color=TEXT_COLOR, line=dict(width=2, color=ACCENT_COLOR)))
    f4.update_layout(**pl_layout)
    figs.append(f4)

    # [5] Scatter Plot: Range Efficiency over Years
    f5 = px.scatter(dff.sample(min(800, len(dff))), x='Model Year', y='Electric Range', title="5. Range vs Year (Sampled)", opacity=0.7, color_discrete_sequence=[SECONDARY_COLOR])
    f5.update_layout(**pl_layout)
    figs.append(f5)

    # [6] Histogram: Range Distribution
    f6 = px.histogram(dff.sample(min(5000, len(dff))), x='Electric Range', nbins=30, title="6. Range Distribution", color_discrete_sequence=[TERTIARY_COLOR])
    f6.update_layout(**pl_layout)
    figs.append(f6)

    # [7] Box Plot: Range Variability by Top Makes
    f7 = px.box(dff_top.sample(min(5000, len(dff_top))), x='Make', y='Electric Range', title="7. Range Spread by Make", color='Make')
    f7.update_layout(**pl_layout, showlegend=False)
    figs.append(f7)

    # [8] Area Chart: Avg Range Progression
    avg_rng = dff.groupby('Model Year')['Electric Range'].mean().reset_index()
    f8 = px.area(avg_rng, x='Model Year', y='Electric Range', title="8. Avg Range Progression")
    f8.update_traces(line_color=TERTIARY_COLOR, fillcolor='rgba(78, 205, 196, 0.4)')
    f8.update_layout(**pl_layout)
    figs.append(f8)

    # [9] Heatmap: Year vs. Range Density
    f9 = px.density_heatmap(dff.sample(min(5000, len(dff))), x='Model Year', y='Electric Range', title="9. Density: Year vs Range", color_continuous_scale='Magma')
    f9.update_layout(**pl_layout, coloraxis_showscale=False)
    figs.append(f9)

    # [10] Lollipop Chart: Avg Range by Manufacturer
    rng_make = dff_top.groupby('Make')['Electric Range'].mean().sort_values().reset_index()
    f10 = go.Figure()
    f10.add_trace(go.Scatter(x=rng_make['Electric Range'], y=rng_make['Make'], mode='markers', marker=dict(color=ACCENT_COLOR, size=14, symbol='diamond')))
    for _, row in rng_make.iterrows():
        f10.add_shape(type='line', x0=0, x1=row['Electric Range'], y0=row['Make'], y1=row['Make'], line=dict(color=HIGHLIGHT_COLOR, width=3))
    f10.update_layout(title="10. Avg Range Per Make", xaxis_title="Miles", **pl_layout)
    figs.append(f10)

    # [11] Donut Chart: Top 5 Active Models
    if 'Model' in dff.columns:
        tm = dff['Model'].value_counts().nlargest(5).reset_index()
        tm.columns = ['Model', 'Count']
        f11 = px.pie(tm, names='Model', values='Count', title="11. Top 5 Models", hole=0.6, color_discrete_sequence=px.colors.qualitative.Prism)
        f11.update_traces(textposition='inside', textinfo='percent+label', marker=dict(line=dict(color=CARD_COLOR, width=2)))
        f11.update_layout(**pl_layout, showlegend=False)
        figs.append(f11)
    else: figs.append(go.Figure().update_layout(title="11. Top Models (N/A)", **pl_layout))

    # [12] Advanced Widget: Growth Delta Indicator
    curr_yr = year_counts['Model Year'].max()
    prev_yr = curr_yr - 1
    c_curr = year_counts[year_counts['Model Year'] == curr_yr]['Count'].sum() if curr_yr in year_counts['Model Year'].values else 0
    c_prev = year_counts[year_counts['Model Year'] == prev_yr]['Count'].sum() if prev_yr in year_counts['Model Year'].values else 0
    
    f12 = go.Figure(go.Indicator(
        mode="gauge+number+delta",
        value=c_curr,
        title={'text': f"12. {curr_yr} Volume YoY", 'font': {'size': 20, 'color': TEXT_COLOR}},
        delta={'reference': c_prev, 'increasing': {'color': "#2ECC71"}},
        gauge={
            'axis': {'range': [None, max(c_curr, c_prev, 1)*1.5]},
            'bar': {'color': ACCENT_COLOR},
            'bgcolor': '#212F3D', 'borderwidth': 0
        }
    ))
    f12.update_layout(**pl_layout)
    figs.append(f12)

    return figs

def build_map_and_insights(dff):
    # Extract lat/lon from Vehicle Location (assuming POINT (lon lat) format)
    dff = dff.copy()
    loc_pattern = r'POINT \(([^ ]+) ([^)]+)\)'
    extracted = dff['Vehicle Location'].str.extract(loc_pattern)
    dff['lon'] = pd.to_numeric(extracted[0], errors='coerce')
    dff['lat'] = pd.to_numeric(extracted[1], errors='coerce')
    dff = dff.dropna(subset=['lat', 'lon'])
    
    # World Satellite Map
    sample_size = min(10000, len(dff))
    map_fig = px.scatter_mapbox(
        dff.sample(sample_size), 
        lat='lat', 
        lon='lon', 
        hover_name='Make', 
        hover_data=['Model', 'Model Year', 'Electric Range'],
        mapbox_style='open-street-map',
        zoom=3,
        title="EV Locations Worldwide (Map View)"
    )
    map_fig.update_layout(
        height=800, 
        width=None, 
        paper_bgcolor='white', 
        plot_bgcolor='white',
        margin=dict(l=0, r=0, t=40, b=0)
    )
    
    # Insights in a styled box
    total_evs = len(dff)
    top_make = dff['Make'].value_counts().index[0] if total_evs > 0 else "N/A"
    avg_range = dff['Electric Range'].mean()
    insights_html = f"""
    <div style="
        position: absolute; 
        top: 50px; 
        right: 20px; 
        width: 300px; 
        background: rgba(255, 255, 255, 0.9); 
        border: 2px solid #00D4FF; 
        border-radius: 10px; 
        padding: 15px; 
        box-shadow: 0 4px 12px rgba(0,0,0,0.3); 
        font-family: 'Segoe UI', sans-serif; 
        z-index: 1000;
    ">
        <div style="color: #00D4FF; font-weight: bold; margin-bottom: 10px; font-size: 16px;">🚀 Key Conclusions</div>
        <div style="color: #333; font-size: 14px; line-height: 1.5;">
            <ul style="margin: 0; padding-left: 20px;">
                <li><strong>Total EVs:</strong> {total_evs:,} vehicles registered.</li>
                <li><strong>Top Manufacturer:</strong> {top_make} leads the market.</li>
                <li><strong>Avg Range:</strong> {avg_range:.1f} miles.</li>
                <li><strong>Trend:</strong> Rapid adoption worldwide.</li>
            </ul>
        </div>
        <div style="color: #00D4FF; font-weight: bold; margin-top: 15px; margin-bottom: 10px; font-size: 16px;">💡 Important Messages</div>
        <div style="color: #333; font-size: 14px; line-height: 1.5;">
            <ul style="margin: 0; padding-left: 20px;">
                <li>EV infrastructure is expanding globally.</li>
                <li>Battery tech drives range improvements.</li>
                <li>Policy support accelerates adoption.</li>
                <li>Address charging network gaps.</li>
            </ul>
        </div>
    </div>
    """
    
    return map_fig, insights_html

# ---------------------------------------------------------
# 4. NAVIGATION AND LAYOUT
# ---------------------------------------------------------
tab_contents = ['Overview (1-4)', 'Trends (5-8)', 'Analysis (9-12)', 'Map & Insights']
tab_buttons = [widgets.Button(description=tab, layout=widgets.Layout(width='auto')) for tab in tab_contents]
current_tab = 0

def switch_tab(b):
    global current_tab
    current_tab = tab_buttons.index(b)
    update_display()

for btn in tab_buttons:
    btn.on_click(switch_tab)

tab_bar = widgets.HBox(tab_buttons, layout=widgets.Layout(justify_content='center', margin='20px 0'))

display_area = widgets.Output()

def update_display():
    with display_area:
        display_area.clear_output(wait=True)
        if current_tab == 0:  # Overview
            grid = widgets.GridBox(chart_outputs[:4], layout=widgets.Layout(width='100%', grid_template_columns="repeat(2, 1fr)", grid_gap='20px'))
            display(grid)
        elif current_tab == 1:  # Trends
            grid = widgets.GridBox(chart_outputs[4:8], layout=widgets.Layout(width='100%', grid_template_columns="repeat(2, 1fr)", grid_gap='20px'))
            display(grid)
        elif current_tab == 2:  # Analysis
            grid = widgets.GridBox(chart_outputs[8:12], layout=widgets.Layout(width='100%', grid_template_columns="repeat(2, 1fr)", grid_gap='20px'))
            display(grid)
        elif current_tab == 3:  # Map & Insights
            map_fig, insights = build_map_and_insights(df)
            with display_area:
                display_area.clear_output(wait=True)
                map_fig.show(config={'scrollZoom': True, 'displayModeBar': True, 'modeBarButtonsToRemove': ['pan2d', 'lasso2d']})
                display(HTML(insights))

# ---------------------------------------------------------
# 5. SINGLE UPDATE LOOP (CALLBACK)
# ---------------------------------------------------------
def refresh_dashboard(change=None):
    mk, top_n = w_make.value, w_top_n.value
    y_min, y_max = w_year.value
    
    # Filter DataFrame efficiently
    dff = df[(df['Model Year'] >= y_min) & (df['Model Year'] <= y_max)]
    if mk != 'All': 
        dff = dff[dff['Make'] == mk]
        
    update_kpis(dff)
    figs = build_charts(dff, top_n)
    
    # Push updates to output panels cleanly
    for i, out in enumerate(chart_outputs):
        with out:
            out.clear_output(wait=True)
            figs[i].show(config={'displayModeBar': False})
    
    update_display()

w_make.observe(refresh_dashboard, names='value')
w_top_n.observe(refresh_dashboard, names='value')
w_year.observe(refresh_dashboard, names='value')

# ---------------------------------------------------------
# 6. ASSEMBLE FINAL LAYOUT
# ---------------------------------------------------------
left_html = widgets.HTML(f"""
    <div style="background:{BG};border-radius:16px;padding:20px 24px 12px;margin-bottom:8px;
            border-bottom:3px solid {CYAN};display:flex;align-items:center;justify-content:space-between;">
  <div>
    <div style="font-family:'DM Sans',sans-serif;font-size:35px;font-weight:700;
                color:{TEXT_PRI};letter-spacing:-0.5px;">
      EV <span style="color:{CYAN};">Analytics</span> Dashboard
    </div>
    <div style="color:{TEXT_SEC};font-size:12px;font-family:'DM Sans',sans-serif;margin-top:3px;">
 · Multi-dimensional Analysis ·
    </div>
  </div>
</div>
    <div class='section-title' ><span style="color:{CYAN};">🎛️ Filter Console</span></div>
""")

left_panel = widgets.VBox(
    [left_html, w_make, w_top_n, w_year, widgets.HTML("<div class='section-title'>📈 Performance Metrics</div>"), kpi_html],
    layout=widgets.Layout(width='30%', height='100%' , padding='20px', border_right=f'2px solid {CYAN}')
)

right_panel = widgets.VBox([tab_bar, display_area], layout=widgets.Layout(width='70%', padding='25px'))

main_dashboard = widgets.HBox(
    [left_panel, right_panel], 
    layout=widgets.Layout(width='100%', background_color=BG_COLOR,  border_radius='15px')
)

BG_DASH = "#020617"   # 🔵 Dark blue background (recommended)

main_dashboard = widgets.HBox(
    [left_panel, right_panel],
    layout=widgets.Layout(
        width='100%',
        height='100%',
        padding='15px',
        background_color=BG_DASH   # ✅ MAIN BACKGROUND
    )
)
from IPython.display import HTML, display

display(HTML("""
<style>
body {
    background-color: #020617 !important;
}
</style>
"""))
display(HTML("""
<style>
.dashboard-container {
    background: linear-gradient(135deg, #020617, #0f172a);
}
</style>
"""))

main_dashboard.add_class("dashboard-container")

# Initialize and render
refresh_dashboard()
display(main_dashboard)

Loading Electric Vehicle Population Data...
Data loaded successfully!
